# Tool 8: analyze_spatial_landcover_terrain (Categorical LULC & Topographic Landscape Engine)

This tool performs **categorical Land-Use / Land-Cover (LULC) surface accounting**, **landscape patch fragmentation metrics (8-connectivity)**, and **continuous DEM elevation/slope profiling**. It also powers the **Tool 7 to Tool 8 handshake** via zonal cross-tabulation (evaluating what specific land-cover classes and terrain slope bands were affected by a detected change mask).

In [ ]:
# 1. Install Dependencies
!pip -q install rasterio numpy scipy pydantic fastmcp

In [ ]:
# 2. Core Imports
import io
import json
import os
import sys
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass
from typing import Dict, List, Literal, Optional, Tuple, Union, Any

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_bounds
from scipy import ndimage
from pydantic import BaseModel, Field, field_validator
from fastmcp import FastMCP

In [ ]:
# 3. Request Validation Model
DEFAULT_WORLDCOVER_LEGEND = {
    10: "Tree cover",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up",
    60: "Bare / sparse vegetation",
    70: "Snow and ice",
    80: "Permanent water bodies",
    90: "Herbaceous wetland",
    95: "Mangroves",
    100: "Moss and lichen",
}

class SpatialLandcoverTerrainRequest(BaseModel):
    lulc_raster_path: str = Field(..., description="Path to classified land-cover GeoTIFF raster")
    dem_raster_path: Optional[str] = Field(default=None, description="Optional path to DEM elevation GeoTIFF")
    zone_mask_path: Optional[str] = Field(default=None, description="Optional path to zone mask GeoTIFF")
    class_legend: Optional[Dict[int, str]] = Field(default=None, description="Custom class mapping dictionary")
    zone_legend: Optional[Dict[int, str]] = Field(default=None, description="Custom zone mapping dictionary")
    calculate_fragmentation: bool = Field(default=True, description="Calculate landscape patch metrics")
    output_dir: Optional[str] = Field(default=None, description="Custom output directory")

    @field_validator("lulc_raster_path")
    @classmethod
    def validate_lulc_path(cls, value: str) -> str:
        p = Path(value)
        if not p.exists() or not p.is_file():
            raise ValueError(f"LULC raster file does not exist: '{value}'")
        return str(p.resolve())

    @field_validator("dem_raster_path", "zone_mask_path")
    @classmethod
    def validate_optional_paths(cls, value: Optional[str]) -> Optional[str]:
        if value is not None:
            p = Path(value)
            if not p.exists() or not p.is_file():
                raise ValueError(f"Specified raster file does not exist: '{value}'")
            return str(p.resolve())
        return None

In [ ]:
# 4. Import Core Engine
from Tool_8_analyze_spatial_landcover_terrain.analyze_spatial_landcover_terrain import analyze_spatial_landcover_terrain
print('Tool 8 engine imported successfully!')

In [ ]:
# 5. FastMCP Tool Registration
mcp = FastMCP("Tool 8: Spatial Landcover & Terrain Analysis Server")

@mcp.tool(
    name="analyze_spatial_landcover_terrain",
    description="Analyze categorical LULC land cover composition, patch fragmentation metrics, DEM elevation and terrain slope profiles, with zonal cross-tabulation against Tool 7 change masks.",
)
def analyze_spatial_landcover_terrain_mcp(
    lulc_raster_path: str,
    dem_raster_path: Optional[str] = None,
    zone_mask_path: Optional[str] = None,
    calculate_fragmentation: bool = True,
    output_dir: Optional[str] = None,
) -> dict:
    try:
        req = SpatialLandcoverTerrainRequest(
            lulc_raster_path=lulc_raster_path,
            dem_raster_path=dem_raster_path,
            zone_mask_path=zone_mask_path,
            calculate_fragmentation=calculate_fragmentation,
            output_dir=output_dir,
        )
        return analyze_spatial_landcover_terrain(req)
    except ValueError as e:
        return {"status": "error", "error": {"type": "validation_error", "message": str(e)}}
    except Exception as e:
        return {"status": "error", "error": {"type": "service_error", "message": str(e)}}

In [ ]:
# 6. Demonstration with Sample Input
sample_in = Path('sample_input.json')
if sample_in.exists():
    with open(sample_in, 'r') as f:
        req = SpatialLandcoverTerrainRequest(**json.load(f))
    res = analyze_spatial_landcover_terrain(req)
    print('Status:', res.get('status'))
    print('Dominant Landcover:', res.get('dominant_landcover'))
    print('Total Valid AOI (km2):', res['spatial_coverage']['valid_aoi_area_km2'])
    if res.get('terrain_profile'):
        print('Mean Elevation (m):', res['terrain_profile']['elevation_meters']['mean_elevation_m'])
        print('Mean Slope (deg):', res['terrain_profile']['slope_profile']['mean_slope_deg'])
    if res.get('zonal_cross_tabulation'):
        print('Zonal Cross-Tabulation Zones:', list(res['zonal_cross_tabulation'].keys()))